In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

# training matches
df = pd.read_parquet(PROCESSED_DIR / "matches_clean.parquet")
df = df.sort_values("date").reset_index(drop=True)

# elo history (pre-match ratings for every match)
elo_df = pd.read_parquet(PROCESSED_DIR / "elo_history.parquet")

print(f"Matches: {len(df)}")
print(f"Elo history rows: {len(elo_df)}")

C:\Users\vasan\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Matches: 32287
Elo history rows: 32287


In [2]:
def compute_form(df, window=5):
    """
    For each team, compute rolling stats BEFORE each match.
    Returns dict: (date, home_team, away_team) -> form stats
    """
    # build per-team match history
    records = []
    for _, row in df.iterrows():
        records.append({
            "date": row["date"], "team": row["home_team"],
            "goals_for": row["home_score"], "goals_against": row["away_score"],
            "outcome": row["outcome"], "is_home": True,
            "tournament": row["tournament"]
        })
        records.append({
            "date": row["date"], "team": row["away_team"],
            "goals_for": row["away_score"], "goals_against": row["home_score"],
            "outcome": row["outcome"], "is_home": False,
            "tournament": row["tournament"]
        })

    team_df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)

    # compute rolling stats per team
    form = {}
    for team, group in team_df.groupby("team"):
        group = group.sort_values("date").reset_index(drop=True)
        for i, row in group.iterrows():
            past = group.iloc[max(0, i - window):i]
            if len(past) == 0:
                form[(row["date"], team)] = {
                    f"goals_for_{window}":     0.0,
                    f"goals_against_{window}": 0.0,
                    f"win_rate_{window}":      0.0,
                    f"points_{window}":        0.0,
                }
            else:
                wins   = (past["outcome"] == "home_win") & past["is_home"] | \
                         (past["outcome"] == "away_win") & ~past["is_home"]
                draws  = past["outcome"] == "draw"
                points = wins.sum() * 3 + draws.sum()
                form[(row["date"], team)] = {
                    f"goals_for_{window}":     past["goals_for"].mean(),
                    f"goals_against_{window}": past["goals_against"].mean(),
                    f"win_rate_{window}":      wins.mean(),
                    f"points_{window}":        points / len(past),
                }
    return form

print("Computing form features (this takes ~1 minute)...")
form5  = compute_form(df, window=5)
form10 = compute_form(df, window=10)
print("Done!")

Computing form features (this takes ~1 minute)...
Done!


In [3]:
def compute_h2h(df):
    """
    For each match, compute H2H record between the two teams
    using only matches BEFORE this date.
    """
    h2h = {}
    match_list = df[["date","home_team","away_team","outcome"]].values

    for i, (date, home, away, outcome) in enumerate(match_list):
        # all previous meetings between these two teams
        past = [
            m for m in match_list[:i]
            if (m[1] == home and m[2] == away) or
               (m[1] == away and m[2] == home)
        ]

        if len(past) == 0:
            h2h[(date, home, away)] = {
                "h2h_matches":   0,
                "h2h_home_wins": 0.0,
                "h2h_draws":     0.0,
                "h2h_away_wins": 0.0,
            }
        else:
            home_wins = sum(
                1 for m in past if
                (m[1] == home and m[3] == "home_win") or
                (m[1] == away and m[3] == "away_win")
            )
            draws = sum(1 for m in past if m[3] == "draw")
            away_wins = len(past) - home_wins - draws
            h2h[(date, home, away)] = {
                "h2h_matches":   len(past),
                "h2h_home_wins": home_wins / len(past),
                "h2h_draws":     draws / len(past),
                "h2h_away_wins": away_wins / len(past),
            }
    return h2h

print("Computing H2H features (this takes ~2 minutes)...")
h2h = compute_h2h(df)
print("Done!")

Computing H2H features (this takes ~2 minutes)...
Done!


In [4]:
def build_features(df, elo_df, form5, form10, h2h):
    rows = []

    for _, row in df.iterrows():
        date  = row["date"]
        home  = row["home_team"]
        away  = row["away_team"]

        # elo features
        elo_row = elo_df[
            (elo_df["date"] == date) &
            (elo_df["home_team"] == home) &
            (elo_df["away_team"] == away)
        ]
        if len(elo_row) == 0:
            continue

        home_elo = elo_row["home_elo_before"].values[0]
        away_elo = elo_row["away_elo_before"].values[0]
        elo_diff = elo_row["elo_diff"].values[0]

        # form features
        hf5  = form5.get((date, home), {})
        af5  = form5.get((date, away), {})
        hf10 = form10.get((date, home), {})
        af10 = form10.get((date, away), {})

        # h2h features
        h = h2h.get((date, home, away), {
            "h2h_matches": 0, "h2h_home_wins": 0.0,
            "h2h_draws": 0.0, "h2h_away_wins": 0.0
        })

        # tournament weight
        if row["is_worldcup"]:
            tournament_weight = 3
        elif row["is_friendly"]:
            tournament_weight = 1
        else:
            tournament_weight = 2

        rows.append({
            # identifiers
            "date":        date,
            "home_team":   home,
            "away_team":   away,
            "tournament":  row["tournament"],

            # elo
            "home_elo":    home_elo,
            "away_elo":    away_elo,
            "elo_diff":    elo_diff,

            # form 5
            "home_goals_for_5":     hf5.get("goals_for_5", 0),
            "home_goals_against_5": hf5.get("goals_against_5", 0),
            "home_win_rate_5":      hf5.get("win_rate_5", 0),
            "home_points_5":        hf5.get("points_5", 0),
            "away_goals_for_5":     af5.get("goals_for_5", 0),
            "away_goals_against_5": af5.get("goals_against_5", 0),
            "away_win_rate_5":      af5.get("win_rate_5", 0),
            "away_points_5":        af5.get("points_5", 0),

            # form differentials
            "goals_for_diff_5":     hf5.get("goals_for_5", 0) - af5.get("goals_for_5", 0),
            "goals_against_diff_5": hf5.get("goals_against_5", 0) - af5.get("goals_against_5", 0),
            "win_rate_diff_5":      hf5.get("win_rate_5", 0) - af5.get("win_rate_5", 0),
            "points_diff_5":        hf5.get("points_5", 0) - af5.get("points_5", 0),

            # form 10
            "goals_for_diff_10":    hf10.get("goals_for_10", 0) - af10.get("goals_for_10", 0),
            "win_rate_diff_10":     hf10.get("win_rate_10", 0) - af10.get("win_rate_10", 0),
            "points_diff_10":       hf10.get("points_10", 0) - af10.get("points_10", 0),

            # h2h
            "h2h_matches":          h["h2h_matches"],
            "h2h_home_win_rate":    h["h2h_home_wins"],
            "h2h_draw_rate":        h["h2h_draws"],
            "h2h_away_win_rate":    h["h2h_away_wins"],

            # context
            "is_friendly":          int(row["is_friendly"]),
            "is_worldcup":          int(row["is_worldcup"]),
            "tournament_weight":    tournament_weight,
            "neutral":              int(row["neutral"]) if "neutral" in row else 0,

            # target
            "outcome":              row["outcome"],
            "home_score":           row["home_score"],
            "away_score":           row["away_score"],
        })

    return pd.DataFrame(rows)

print("Building feature matrix...")
features_df = build_features(df, elo_df, form5, form10, h2h)
print(f"Feature matrix shape: {features_df.shape}")
print(f"\nFeature columns:")
for col in features_df.columns:
    print(f"  {col}")

Building feature matrix...
Feature matrix shape: (32287, 33)

Feature columns:
  date
  home_team
  away_team
  tournament
  home_elo
  away_elo
  elo_diff
  home_goals_for_5
  home_goals_against_5
  home_win_rate_5
  home_points_5
  away_goals_for_5
  away_goals_against_5
  away_win_rate_5
  away_points_5
  goals_for_diff_5
  goals_against_diff_5
  win_rate_diff_5
  points_diff_5
  goals_for_diff_10
  win_rate_diff_10
  points_diff_10
  h2h_matches
  h2h_home_win_rate
  h2h_draw_rate
  h2h_away_win_rate
  is_friendly
  is_worldcup
  tournament_weight
  neutral
  outcome
  home_score
  away_score


In [5]:
print("Missing values:")
print(features_df.isnull().sum()[features_df.isnull().sum() > 0])

print(f"\nOutcome distribution:")
print(features_df["outcome"].value_counts())

print(f"\nSample row:")
print(features_df.iloc[1000][["elo_diff","home_win_rate_5","away_win_rate_5",
                               "h2h_matches","outcome"]].to_string())

# save
out_path = PROCESSED_DIR / "features.parquet"
features_df.to_parquet(out_path, index=False)
print(f"\nSaved feature matrix → {out_path}")
print("Next: 05_train_model.ipynb")

Missing values:
Series([], dtype: int64)

Outcome distribution:
outcome
home_win    15652
away_win     9045
draw         7590
Name: count, dtype: int64

Sample row:
elo_diff           32.3
home_win_rate_5     0.4
away_win_rate_5     0.4
h2h_matches           1
outcome            draw

Saved feature matrix → ..\data\processed\features.parquet
Next: 05_train_model.ipynb
